# Часть A — Базовый блок

Первичный анализ данных и базовая модель Logistic Regression.

**Запуск:** откройте ноутбук в Jupyter и выполните ячейки сверху вниз. Если adult.csv отсутствует, данные автоматически скачиваются из UCI.

## 1. Загрузка данных и подготовка окружения

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report, ConfusionMatrixDisplay)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

RANDOM_STATE = 42
DATA_PATH = Path('adult.csv')
if not DATA_PATH.exists():
    import urllib.request
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
    columns = ['age','workclass','fnlwgt','education','education.num','marital.status','occupation',
      'relationship','race','sex','capital.gain','capital.loss','hours.per.week','native.country','income']
    raw = pd.read_csv(url, header=None, names=columns, na_values='?', skipinitialspace=True)
    raw.to_csv(DATA_PATH, index=False)
    print('Датасет загружен из UCI:', DATA_PATH)

df = pd.read_csv(DATA_PATH, na_values='?', skipinitialspace=True)
df.columns = df.columns.str.strip()
df['income'] = df['income'].astype(str).str.strip().str.replace('.', '', regex=False)
df['income'] = (df['income'] == '>50K').astype(int)
print('Размер:', df.shape)
display(df.head())
df.info()


## 2. Базовые задания
- Пропуски и их доли.
- Кодирование целевого признака income.
- Разделение train/test со стратификацией.
- Заполнение пропусков, One-Hot Encoding и масштабирование внутри Pipeline.
- Logistic Regression, Accuracy и матрица ошибок.

In [ ]:
missing = pd.DataFrame({'missing_count':df.isna().sum(),'missing_percent':(df.isna().mean()*100).round(2)})
display(missing.sort_values('missing_count',ascending=False))
print('Дубликаты:', df.duplicated().sum())
display(df['income'].value_counts().rename(index={0:'<=50K',1:'>50K'}).to_frame('count'))
print('Доли классов:')
display((df['income'].value_counts(normalize=True)*100).round(2))
X, y = df.drop(columns='income'), df['income']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.2,stratify=y,random_state=RANDOM_STATE)
# education — текстовый дубль education.num, исключаем его до обучения.
X_train = X_train.drop(columns=['education'], errors='ignore')
X_test = X_test.drop(columns=['education'], errors='ignore')
cat_cols = X_train.select_dtypes(include=['object','category']).columns.tolist()
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
num_pipe = Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
cat_pipe = Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer([('num',num_pipe,num_cols),('cat',cat_pipe,cat_cols)])
baseline = Pipeline([('prep',preprocessor),('model',LogisticRegression(max_iter=2000,random_state=RANDOM_STATE))])
baseline.fit(X_train,y_train)
pred = baseline.predict(X_test)
print('Baseline accuracy:',round(accuracy_score(y_test,pred),4))
print(classification_report(y_test,pred,target_names=['<=50K','>50K']))
ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=['<=50K','>50K'])
plt.show()


## 3. Первичный EDA

In [ ]:
display(X_train[num_cols].describe().T)
for col in ['age','hours.per.week','education.num']:
    if col in X_train:
        sns.histplot(data=pd.DataFrame({'value':X_train[col],'income':y_train}),x='value',hue='income',bins=30,stat='density',common_norm=False)
        plt.title(col); plt.show()
for col in ['workclass','marital.status','occupation']:
    if col in X_train:
        display(pd.crosstab(X_train[col].fillna('Missing'),y_train,normalize='index').round(3))
